# Día 2 · Evaluación e interpretabilidad de modelos

En auditoría, un modelo se analiza por rendimiento y por explicabilidad.

In [ ]:
import numpy as np
import pandas as pd

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def generar_dataset_bancario(
    n=5000,
    seed=42,
    incluir_leakage=False,
    incluir_missing=True,
    incluir_outliers=True
):
    rng = np.random.default_rng(seed)

    cliente_id = np.arange(1, n + 1)
    edad = np.clip(rng.normal(42, 12, n).round(), 18, 85).astype(int)
    antiguedad_cliente = np.clip(
        (edad - 18) * rng.uniform(0.05, 0.6, n) + rng.normal(2, 3, n),
        0, 40
    ).round(1)
    ingresos_mensuales = np.clip(
        rng.lognormal(mean=np.log(2200), sigma=0.45, size=n),
        700, 15000
    ).round(2)

    situacion_laboral = rng.choice(
        ["asalariado", "autonomo", "temporal", "desempleado", "jubilado"],
        size=n,
        p=[0.50, 0.15, 0.15, 0.10, 0.10]
    )
    estado_civil = rng.choice(
        ["soltero", "casado", "divorciado", "viudo"],
        size=n, p=[0.38, 0.42, 0.15, 0.05]
    )
    nivel_estudios = rng.choice(
        ["basicos", "medios", "universitarios", "postgrado"],
        size=n, p=[0.25, 0.35, 0.30, 0.10]
    )
    provincia_riesgo = rng.choice(
        ["bajo", "medio", "alto"],
        size=n, p=[0.45, 0.40, 0.15]
    )

    num_productos = rng.choice([1, 2, 3, 4, 5, 6], size=n, p=[0.18, 0.26, 0.23, 0.17, 0.10, 0.06])
    saldo_medio_3m = np.clip(
        ingresos_mensuales * rng.normal(1.2, 0.7, n) + rng.normal(0, 500, n),
        -3000, 50000
    ).round(2)
    limite_credito = np.clip(
        ingresos_mensuales * rng.normal(3.5, 1.2, n), 500, 40000
    ).round(2)
    deuda_total = np.clip(
        ingresos_mensuales * rng.normal(4.5, 2.0, n) + rng.normal(0, 2000, n),
        0, 120000
    ).round(2)
    cuota_mensual_prestamos = np.clip(
        deuda_total * rng.uniform(0.01, 0.04, n), 0, 3500
    ).round(2)
    utilizacion_credito = np.clip(
        rng.beta(2, 3, n) + rng.normal(0, 0.08, n), 0, 1
    ).round(3)
    porcentaje_ahorro = np.clip(
        saldo_medio_3m / np.maximum(ingresos_mensuales, 1), -1.5, 8
    ).round(3)
    ratio_endeudamiento = np.clip(
        (cuota_mensual_prestamos / np.maximum(ingresos_mensuales, 1)), 0, 2
    ).round(3)

    retrasos_30d_12m = rng.poisson(0.8, n)
    retrasos_60d_12m = rng.poisson(0.25, n)
    retrasos_90d_12m = rng.poisson(0.08, n)

    retrasos_30d_12m += np.where(situacion_laboral == "temporal", rng.integers(0, 2, n), 0)
    retrasos_30d_12m += np.where(situacion_laboral == "desempleado", rng.integers(1, 3, n), 0)
    retrasos_60d_12m += np.where(situacion_laboral == "desempleado", rng.integers(0, 2, n), 0)
    retrasos_90d_12m += np.where(situacion_laboral == "desempleado", rng.integers(0, 2, n), 0)

    impagos_previos = rng.binomial(1, 0.07, n)
    consultas_credito_6m = rng.poisson(1.8, n)
    nuevas_cuentas_12m = rng.poisson(0.9, n)
    uso_banca_digital = np.clip(rng.normal(0.65, 0.22, n), 0, 1).round(3)
    alertas_fraude_12m = rng.binomial(3, 0.04, n)
    reclamaciones_12m = rng.poisson(0.35, n)

    segmento_cliente = np.select(
        [
            ingresos_mensuales < 1400,
            (ingresos_mensuales >= 1400) & (ingresos_mensuales < 3000),
            ingresos_mensuales >= 3000
        ],
        ["bajo", "medio", "alto"],
        default="medio"
    )

    tiene_hipoteca = rng.binomial(1, np.clip((edad - 25) / 60, 0.05, 0.55))
    tiene_prestamo_personal = rng.binomial(1, 0.28, n)
    tiene_tarjeta_credito = rng.binomial(1, 0.72, n)
    tiene_seguro = rng.binomial(1, 0.48, n)

    score_base = (
        700
        + 25 * np.log1p(np.maximum(ingresos_mensuales, 1))
        + 8 * antiguedad_cliente
        - 90 * ratio_endeudamiento
        - 35 * utilizacion_credito
        - 18 * retrasos_30d_12m
        - 35 * retrasos_60d_12m
        - 60 * retrasos_90d_12m
        - 70 * impagos_previos
        - 12 * consultas_credito_6m
        + 10 * num_productos
        + rng.normal(0, 30, n)
    )
    score_interno = np.clip(score_base, 300, 950).round().astype(int)

    logit_default = (
        -4.2
        + 2.8 * ratio_endeudamiento
        + 1.2 * utilizacion_credito
        + 0.22 * retrasos_30d_12m
        + 0.55 * retrasos_60d_12m
        + 0.95 * retrasos_90d_12m
        + 1.10 * impagos_previos
        + 0.10 * consultas_credito_6m
        + 0.18 * nuevas_cuentas_12m
        + 0.45 * (situacion_laboral == "temporal").astype(int)
        + 1.05 * (situacion_laboral == "desempleado").astype(int)
        + 0.35 * (provincia_riesgo == "alto").astype(int)
        - 0.00022 * ingresos_mensuales
        - 0.03 * antiguedad_cliente
        - 0.0025 * score_interno
        - 0.15 * (num_productos >= 3).astype(int)
        + 0.28 * alertas_fraude_12m
        + rng.normal(0, 0.45, n)
    )

    prob_default = sigmoid(logit_default)
    default = rng.binomial(1, prob_default)

    dias_mora_actual = np.where(
        default == 1,
        rng.choice([15, 30, 45, 60, 90, 120], size=n, p=[0.10, 0.20, 0.20, 0.20, 0.20, 0.10]),
        0
    )
    recobro_gestionado = np.where(default == 1, rng.binomial(1, 0.35, n), 0)
    refinanciado_post = np.where(default == 1, rng.binomial(1, 0.22, n), 0)

    df = pd.DataFrame({
        "cliente_id": cliente_id,
        "edad": edad,
        "antiguedad_cliente": antiguedad_cliente,
        "ingresos_mensuales": ingresos_mensuales,
        "situacion_laboral": situacion_laboral,
        "estado_civil": estado_civil,
        "nivel_estudios": nivel_estudios,
        "provincia_riesgo": provincia_riesgo,
        "segmento_cliente": segmento_cliente,
        "num_productos": num_productos,
        "saldo_medio_3m": saldo_medio_3m,
        "limite_credito": limite_credito,
        "deuda_total": deuda_total,
        "cuota_mensual_prestamos": cuota_mensual_prestamos,
        "utilizacion_credito": utilizacion_credito,
        "porcentaje_ahorro": porcentaje_ahorro,
        "ratio_endeudamiento": ratio_endeudamiento,
        "retrasos_30d_12m": retrasos_30d_12m,
        "retrasos_60d_12m": retrasos_60d_12m,
        "retrasos_90d_12m": retrasos_90d_12m,
        "impagos_previos": impagos_previos,
        "consultas_credito_6m": consultas_credito_6m,
        "nuevas_cuentas_12m": nuevas_cuentas_12m,
        "uso_banca_digital": uso_banca_digital,
        "alertas_fraude_12m": alertas_fraude_12m,
        "reclamaciones_12m": reclamaciones_12m,
        "tiene_hipoteca": tiene_hipoteca,
        "tiene_prestamo_personal": tiene_prestamo_personal,
        "tiene_tarjeta_credito": tiene_tarjeta_credito,
        "tiene_seguro": tiene_seguro,
        "score_interno": score_interno,
        "default": default
    })

    if incluir_leakage:
        df["dias_mora_actual"] = dias_mora_actual
        df["recobro_gestionado"] = recobro_gestionado
        df["refinanciado_post"] = refinanciado_post

    if incluir_outliers:
        idx_outliers = rng.choice(df.index, size=max(1, int(0.01 * n)), replace=False)
        df.loc[idx_outliers, "ingresos_mensuales"] *= rng.uniform(3, 8, size=len(idx_outliers))
        df.loc[idx_outliers, "deuda_total"] *= rng.uniform(2, 5, size=len(idx_outliers))
        df.loc[idx_outliers, "saldo_medio_3m"] *= rng.uniform(-2, 4, size=len(idx_outliers))
        df["ingresos_mensuales"] = df["ingresos_mensuales"].round(2)
        df["deuda_total"] = df["deuda_total"].round(2)
        df["saldo_medio_3m"] = df["saldo_medio_3m"].round(2)

    if incluir_missing:
        columnas_missing = {
            "ingresos_mensuales": 0.03,
            "saldo_medio_3m": 0.04,
            "ratio_endeudamiento": 0.02,
            "uso_banca_digital": 0.03,
            "nivel_estudios": 0.02,
            "estado_civil": 0.015
        }
        for col, pct in columnas_missing.items():
            idx = rng.choice(df.index, size=int(n * pct), replace=False)
            df.loc[idx, col] = np.nan

    return df

def preparar_X_y(df):
    cols_drop = ["cliente_id", "default", "dias_mora_actual", "recobro_gestionado", "refinanciado_post"]
    X = df.drop(columns=[c for c in cols_drop if c in df.columns])
    y = df["default"].copy()
    return X, y

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, brier_score_loss,
    classification_report, RocCurveDisplay
)
from sklearn.calibration import calibration_curve

try:
    import shap
except Exception:
    shap = None

In [ ]:
df = generar_dataset_bancario(n=4500, seed=555, incluir_leakage=False)
X, y = preparar_X_y(df)

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
    ]), cat_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

logit = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=5000))
])
rf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    ))
])

logit.fit(X_train, y_train)
rf.fit(X_train, y_train)

In [ ]:
for nombre, modelo in [("Logit", logit), ("Random Forest", rf)]:
    pred = modelo.predict(X_test)
    proba = modelo.predict_proba(X_test)[:, 1]
    print("\n", "="*50)
    print(nombre)
    print(classification_report(y_test, pred))
    print("ROC AUC:", round(roc_auc_score(y_test, proba), 4))
    print("Brier score:", round(brier_score_loss(y_test, proba), 4))

In [ ]:
RocCurveDisplay.from_predictions(y_test, logit.predict_proba(X_test)[:, 1], name="Logit")
RocCurveDisplay.from_predictions(y_test, rf.predict_proba(X_test)[:, 1], name="Random Forest")
plt.title("Comparación ROC")
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
for nombre, modelo in [("Logit", logit), ("Random Forest", rf)]:
    proba = modelo.predict_proba(X_test)[:, 1]
    frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=10)
    plt.plot(mean_pred, frac_pos, marker="o", label=nombre)

plt.plot([0, 1], [0, 1], "--")
plt.xlabel("Probabilidad predicha media")
plt.ylabel("Frecuencia observada")
plt.title("Curva de calibración")
plt.legend()
plt.show()

In [ ]:
feature_names = rf.named_steps["preprocessor"].get_feature_names_out()
importancias = rf.named_steps["model"].feature_importances_

imp_df = pd.DataFrame({
    "variable": feature_names,
    "importancia": importancias
}).sort_values("importancia", ascending=False).head(20)

imp_df.plot.barh(x="variable", y="importancia", figsize=(8, 6))
plt.gca().invert_yaxis()
plt.title("Top variables Random Forest")
plt.show()

In [ ]:
if shap is None:
    print("Instala shap con: pip install shap")
else:
    Xt = rf.named_steps["preprocessor"].transform(X_test)
    explainer = shap.TreeExplainer(rf.named_steps["model"])
    shap_values = explainer.shap_values(Xt[:300])
    shap.summary_plot(shap_values[1], Xt[:300], feature_names=feature_names, show=False)
    plt.show()

## Ejercicios propuestos

    1. Evaluación e interpretabilidad: ejercicio 1. Define el objetivo, programa la solución y justifica el resultado.
2. Evaluación e interpretabilidad: ejercicio 2. Define el objetivo, programa la solución y justifica el resultado.
3. Evaluación e interpretabilidad: ejercicio 3. Define el objetivo, programa la solución y justifica el resultado.
4. Evaluación e interpretabilidad: ejercicio 4. Define el objetivo, programa la solución y justifica el resultado.
5. Evaluación e interpretabilidad: ejercicio 5. Define el objetivo, programa la solución y justifica el resultado.
6. Evaluación e interpretabilidad: ejercicio 6. Define el objetivo, programa la solución y justifica el resultado.
7. Evaluación e interpretabilidad: ejercicio 7. Define el objetivo, programa la solución y justifica el resultado.
8. Evaluación e interpretabilidad: ejercicio 8. Define el objetivo, programa la solución y justifica el resultado.
9. Evaluación e interpretabilidad: ejercicio 9. Define el objetivo, programa la solución y justifica el resultado.
10. Evaluación e interpretabilidad: ejercicio 10. Define el objetivo, programa la solución y justifica el resultado.

## Solucionario orientativo

    ### Solución 1
Pauta sugerida: plantea el split, construye el pipeline, entrena, evalúa con ROC AUC y comenta qué ocurre.
### Solución 2
Pauta sugerida: plantea el split, construye el pipeline, entrena, evalúa con ROC AUC y comenta qué ocurre.
### Solución 3
Pauta sugerida: plantea el split, construye el pipeline, entrena, evalúa con ROC AUC y comenta qué ocurre.
### Solución 4
Pauta sugerida: plantea el split, construye el pipeline, entrena, evalúa con ROC AUC y comenta qué ocurre.
### Solución 5
Pauta sugerida: plantea el split, construye el pipeline, entrena, evalúa con ROC AUC y comenta qué ocurre.
### Solución 6
Pauta sugerida: plantea el split, construye el pipeline, entrena, evalúa con ROC AUC y comenta qué ocurre.
### Solución 7
Pauta sugerida: plantea el split, construye el pipeline, entrena, evalúa con ROC AUC y comenta qué ocurre.
### Solución 8
Pauta sugerida: plantea el split, construye el pipeline, entrena, evalúa con ROC AUC y comenta qué ocurre.
### Solución 9
Pauta sugerida: plantea el split, construye el pipeline, entrena, evalúa con ROC AUC y comenta qué ocurre.
### Solución 10
Pauta sugerida: plantea el split, construye el pipeline, entrena, evalúa con ROC AUC y comenta qué ocurre.